# Scaling ML Workflows: PySpark ETL on WikiText & Cloud TPU miniGPT Training

### An end-to-end distributed Machine Learning pipeline on Google Kubernetes Engine (GKE)

This notebook demonstrates scaling an ML workflow from local debugging to full multi-node distribution without leaving the Jupyter interface. We will:

1. Use **Apache Spark** to distribute the preprocessing of a WikiText dataset into chunked Token IDs.
2. Prototype a **miniGPT Language Model in JAX/Flax** on our local Jupyter TPU slices.
3. Scale out to a **Multi-Host TPU v5p Slice** for data-parallel JAX training using Kubeflow Trainer.
4. Deploy the trained model to an **NVIDIA GPU** backend for high-throughput text generation.

## 0. Setup & environment check

We verify that our Python SDKs are loaded, check that our Workspace is connected to the GKE Cloud TPU hardware, and insert our jobs folder into the Python path.

In [ ]:
import os
import sys
import subprocess
import time

# Ensure the jobs package is on the import path
DEMO_DIR = os.getcwd()
if not os.path.isdir(os.path.join(DEMO_DIR, "jobs")):
    DEMO_DIR = "/home/jovyan/demo"
if DEMO_DIR not in sys.path:
    sys.path.insert(0, DEMO_DIR)

import jax
import kubeflow.trainer
import kubeflow.spark

print("JAX version:", jax.__version__)
print("Local TPU cores detected on this notebook VM:", jax.local_device_count())
print("Local TPU devices:", jax.local_devices())

# Verify the notebook's permissions
for res in ["trainjobs.trainer.kubeflow.org", "sparkapplications.sparkoperator.k8s.io"]:
    out = subprocess.run(["kubectl", "auth", "can-i", "create", res],
                         capture_output=True, text=True)
    print(f"Can-i create {res.split('.')[0]:15s}:", out.stdout.strip() or out.stderr.strip())

### Configure GCS storage

We use a dedicated **GCS bucket** as our shared storage bus. Spark executors will write processed training shards to this bucket, which the JAX TPU trainer will read.

In [ ]:
bucket_name = os.environ.get("DEMO_BUCKET", "sizhang-gke-dev-ml-demo-data")
print(f"Using shared GCS bucket: gs://{bucket_name}")

## Stage 1 — Distributed WikiText Processing with Apache Spark

The WikiText dataset is processed using Spark. The job downloads the raw data, tokenizes it using a HuggingFace GPT2 tokenizer, and generates sequence chunks. The Spark executors write the processed `.npz` shards directly to our shared GCS bucket.

In [ ]:
import os
import shutil
import tempfile
import subprocess
import time

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import Driver, Executor, SparkClient, Name, NodeSelector
from jobs.data_processing import run_etl

NUM_EXECUTORS = 4

# Clean up any stale Spark ETL jobs from previous runs
subprocess.run(["kubectl", "delete", "sparkconnect", "scaling-data-etl", "--ignore-not-found"])

start_time = time.time()

NAMESPACE = os.environ.get("DEMO_NAMESPACE", "default")
REGISTRY = os.environ.get("REGISTRY", "us-west1-docker.pkg.dev/sizhang-gke-dev/sizhang-repo")
TAG = os.environ.get("TAG", "local-gke-dev")
SPARK_IMAGE = os.environ.get("DEMO_SPARK_IMAGE", f"{REGISTRY}/spark-py311:{TAG}")

# Package jobs directory as zip and send to Spark executors
tmp_dir = tempfile.gettempdir()
zip_path = os.path.join(tmp_dir, "jobs")
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")
shutil.make_archive(zip_path, "zip", root_dir=DEMO_DIR, base_dir="jobs")
zip_file = zip_path + ".zip"

print(f"Connecting to Spark ({NUM_EXECUTORS} executors) via GCS bucket {bucket_name}...")

client = SparkClient(backend_config=KubernetesBackendConfig(namespace=NAMESPACE))
spark = client.connect(
    num_executors=NUM_EXECUTORS,
    driver=Driver(image=SPARK_IMAGE, resources={"cpu": "1", "memory": "4Gi"}),
    executor=Executor(
        num_instances=NUM_EXECUTORS,
        resources_per_executor={"cpu": "1", "memory": "4Gi"},
    ),
    options=[
        Name("scaling-data-etl"),
        NodeSelector({"cloud.google.com/machine-family": "n2"}),
    ],
    spark_conf={
        "spark.kubernetes.container.image": SPARK_IMAGE,
        "spark.kubernetes.driver.label.sidecar.istio.io/inject": "false",
        "spark.kubernetes.executor.label.sidecar.istio.io/inject": "false",
    },
)

spark.addArtifacts(zip_file, pyfile=True)
if os.path.exists(zip_file):
    os.remove(zip_file)

try:
    print(f"Running ETL logic...")
    run_etl(spark, bucket_name=bucket_name, num_records=16000, num_shards=8, block_size=128)
    print("Stage 1 complete.")
finally:
    spark.stop()

end_time = time.time()
print(f"Data processing took {end_time - start_time:.2f} seconds")

In [ ]:
# Print Spark driver logs to verify execution details
# pipeline.print_spark_logs()

## Stage 2 — Local TPU Interactive Debugging

Before executing a massive distributed training run across the cluster, we should verify our model logic. We execute our miniGPT training function locally on the Jupyter Workspace's dedicated TPU cores.

We can adjust the hyperparameters here to train a very small model (e.g., 2 layers) on a subset of the data just to ensure it converges.

In [ ]:
import time

# Set environmental configurations for local debug
os.environ["BUCKET_NAME"] = bucket_name
os.environ["EPOCHS"] = "2"
os.environ["GLOBAL_BATCH_SIZE"] = "8"
os.environ["BLOCK_SIZE"] = "128"
os.environ["VOCAB_SIZE"] = "50257"
os.environ["N_LAYER"] = "2"
os.environ["N_HEAD"] = "2"
os.environ["N_EMBD"] = "64"

from jobs.train import train_scaling_model

start_time = time.time()
# Run locally
train_scaling_model(is_local_debug=True)
end_time = time.time()
print(f"Local debug training took {end_time - start_time:.2f} seconds")

## Stage 3 — Full-scale Distributed TPU Training

Now that our JAX training loop is verified, we are ready to scale out. We will train a larger miniGPT model by submitting a `TrainJob` that spawns across an entire **multi-host TPU v5p slice**.

In [ ]:
import subprocess

# Release TPU nodes held by the reservation placeholder
subprocess.run(["kubectl", "delete", "job", "tpu-job-ccc", "-n", "default", "--ignore-not-found"])

In [ ]:
import os
import time
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import CustomTrainer, TrainerClient
from kubeflow.trainer.options import kubernetes as k8s_options
from jobs.train import train_scaling_model


start_time = time.time()

NAMESPACE = os.environ.get("DEMO_NAMESPACE", "default")
TPU_IMAGE = os.environ.get("DEMO_TPU_IMAGE", "us-docker.pkg.dev/cloud-tpu-images/jax-ai-image/tpu:latest")
NUM_TPU_HOSTS = 2
tpu_compute_class = "tpu-v5-8-multi-host"

def _tpu_placement_patch(compute_class):
    """Return a RuntimePatch that pins pods to TPU nodes."""
    return k8s_options.RuntimePatch(
        training_runtime_spec=k8s_options.TrainingRuntimeSpecPatch(
            template=k8s_options.JobSetTemplatePatch(
                spec=k8s_options.JobSetSpecPatch(
                    replicated_jobs=[
                        k8s_options.ReplicatedJobPatch(
                            name="node",
                            template=k8s_options.JobTemplatePatch(
                                spec=k8s_options.JobSpecPatch(
                                    template=k8s_options.PodTemplatePatch(
                                        spec=k8s_options.PodSpecPatch(
                                            node_selector={"cloud.google.com/compute-class": compute_class},
                                            tolerations=[
                                                {"key": "google.com/tpu", "operator": "Exists", "effect": "NoSchedule"},
                                                {"key": "cloud.google.com/compute-class", "operator": "Exists", "effect": "NoSchedule"},
                                            ],
                                        )
                                    )
                                )
                            )
                        )
                    ]
                )
            )
        )
    )

client = TrainerClient(backend_config=KubernetesBackendConfig(namespace=NAMESPACE))

print(f"Submitting TPU TrainJob ({NUM_TPU_HOSTS} hosts, 4 epochs)...")
job_id = client.train(
    runtime="jax-distributed",
    trainer=CustomTrainer(
        func=train_scaling_model,
        image=TPU_IMAGE,
        num_nodes=NUM_TPU_HOSTS,
        resources_per_node={"google.com/tpu": 4},
        env={
            "JAX_PLATFORMS": "tpu,cpu",
            "ENABLE_PJRT_COMPATIBILITY": "true",
            "BUCKET_NAME": bucket_name,
            "EPOCHS": "4",
            "GLOBAL_BATCH_SIZE": "64",
            "BLOCK_SIZE": "128",
            "VOCAB_SIZE": "50257",
            "N_LAYER": "4",
            "N_HEAD": "4",
            "N_EMBD": "128",
            "LOCAL_DEBUG": "false",
        },
    ),
    options=[_tpu_placement_patch(tpu_compute_class)],
)
print(f"Created training job: {job_id}")

print(f"\n--- Streaming logs for {job_id} ---")
for logline in client.get_job_logs(name=job_id, follow=True):
    print(logline, end="", flush=True)
client.wait_for_job_status(job_id, timeout=3600, polling_interval=10)

end_time = time.time()
print(f"Distributed training took {end_time - start_time:.2f} seconds")

## Stage 4 — GPU-Accelerated Inference & Model Serving

With the final weights (`params.npz`) and metrics in GCS, we can deploy the model. You can deploy your model on a different hardware from the type you trained the model on. 

We submit a Kubernetes Deployment that:
1. Provisions an T4 GPU node.
2. Runs our `serve.py` script.
3. Automatically downloads the Flax weights from GCS and converts them to PyTorch CUDA tensors for high-performance text generation.

In [ ]:
import time
from jobs import pipeline

# Release GPU nodes held by the reservation placeholder
subprocess.run(["kubectl", "delete", "job", "gpu-job-ccc", "-n", "default", "--ignore-not-found"])

start_time = time.time()
# Deploy inference service using the python script helper
pipeline.deploy_inference()
end_time = time.time()
print(f"Inference deployment took {end_time - start_time:.2f} seconds")

### Test predictions

We send an HTTP POST request containing a prompt to our inference service, and the PyTorch backend generates text.

In [ ]:
import urllib.request
import json
import numpy as np
from jobs import PROMPTS

print("=== Generating text from remotely trained larger model ===\n")
for prompt in PROMPTS:
    payload = json.dumps({"prompt": prompt, "max_new_tokens": 50}).encode("utf-8")

    req = urllib.request.Request(
        "http://scaling-model-inference.default.svc.cluster.local:80/generate",
        data=payload,
        headers={"Content-Type": "application/json"}
    )

    try:
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode("utf-8"))
            print("Prompt:", result["prompt"])
            print("Generated Text:", result["text"])
            print("-" * 60)
    except Exception as e:
        print(f"Error querying inference service for prompt '{prompt}': {e}")